In [1]:
import pandas as pd
import numpy as np

# Load historical data from disk
fcf_historical = pd.read_csv('data/fcf_historical.csv', index_col=0).squeeze()
revenue_historical = pd.read_csv('data/revenue_historical.csv', index_col=0).squeeze()

print("Historical Revenue (millions USD):")
print(revenue_historical)
print("\nHistorical FCF (millions USD):")
print(fcf_historical)

Historical Revenue (millions USD):
2023    31352.0
2024    34857.0
2025    37895.0
2026    41525.0
Name: Total Revenue, dtype: float64

Historical FCF (millions USD):
2023     6313.0
2024     9498.0
2025    12434.0
2026    14402.0
Name: 0, dtype: float64


In [2]:
revenue_growth = revenue_historical.pct_change().dropna()
fcf_margin = (fcf_historical / revenue_historical * 100).round(1)

print("Revenue Growth Rate by year:")
print(revenue_growth.apply(lambda x: f"{x:.1%}"))

print("\nFCF Margin by year (FCF as % of Revenue):")
print(fcf_margin.apply(lambda x: f"{x}%"))

print(f"\nAverage revenue growth: {revenue_growth.mean():.1%}")
print(f"Average FCF margin: {fcf_margin.mean():.1f}%")

Revenue Growth Rate by year:
2024    11.2%
2025     8.7%
2026     9.6%
Name: Total Revenue, dtype: str

FCF Margin by year (FCF as % of Revenue):
2023    20.1%
2024    27.2%
2025    32.8%
2026    34.7%
dtype: str

Average revenue growth: 9.8%
Average FCF margin: 28.7%


In [3]:
# Projection assumptions - Base Case
# Years 2027-2031 (5 year forecast period)

projection_years = [2027, 2028, 2029, 2030, 2031]

# Revenue growth rates - decelerating gradually
revenue_growth_proj = {
    2027: 0.090,
    2028: 0.082,
    2029: 0.074,
    2030: 0.066,
    2031: 0.058,
}

# FCF margin - stabilising after recent expansion
fcf_margin_proj = {
    2027: 0.350,
    2028: 0.355,
    2029: 0.358,
    2030: 0.360,
    2031: 0.360,
}

print("Projection Assumptions - Base Case")
print(f"{'Year':<8} {'Revenue Growth':>15} {'FCF Margin':>12}")
print("-" * 37)
for year in projection_years:
    print(f"{year:<8} {revenue_growth_proj[year]:>14.1%} {fcf_margin_proj[year]:>11.1%}")

Projection Assumptions - Base Case
Year      Revenue Growth   FCF Margin
-------------------------------------
2027               9.0%       35.0%
2028               8.2%       35.5%
2029               7.4%       35.8%
2030               6.6%       36.0%
2031               5.8%       36.0%


In [4]:

last_revenue = revenue_historical.loc[2026]

projected_revenue = {}
projected_fcf = {}

for year in projection_years:
    # Each year's revenue grows from the prior year
    if year == 2027:
        projected_revenue[year] = last_revenue * (1 + revenue_growth_proj[year])
    else:
        projected_revenue[year] = projected_revenue[year-1] * (1 + revenue_growth_proj[year])
    
    # FCF = Revenue x FCF Margin
    projected_fcf[year] = projected_revenue[year] * fcf_margin_proj[year]

# Convert to pandas Series for clean display
rev_proj = pd.Series(projected_revenue).round(1)
fcf_proj = pd.Series(projected_fcf).round(1)

print(f"{'Year':<8} {'Revenue ($M)':>14} {'FCF ($M)':>12}")
print("-" * 36)
for year in projection_years:
    print(f"{year:<8} {rev_proj[year]:>14,.1f} {fcf_proj[year]:>12,.1f}")

Year       Revenue ($M)     FCF ($M)
------------------------------------
2027           45,262.2     15,841.8
2028           48,973.8     17,385.7
2029           52,597.8     18,830.0
2030           56,069.3     20,184.9
2031           59,321.3     21,355.7


In [5]:
all_years = list(revenue_historical.index) + projection_years

revenue_combined = pd.concat([revenue_historical, rev_proj])
fcf_combined = pd.concat([fcf_historical, fcf_proj])

# Build display table
summary = pd.DataFrame({
    'Revenue ($M)': revenue_combined,
    'FCF ($M)': fcf_combined,
})

# Label years as historical or projected
summary['Type'] = ['Historical'] * 4 + ['Projected'] * 5

print("Salesforce - Revenue and FCF Model (Base Case)")
print("=" * 55)
print(summary.to_string())

Salesforce - Revenue and FCF Model (Base Case)
      Revenue ($M)  FCF ($M)        Type
2023       31352.0    6313.0  Historical
2024       34857.0    9498.0  Historical
2025       37895.0   12434.0  Historical
2026       41525.0   14402.0  Historical
2027       45262.2   15841.8   Projected
2028       48973.8   17385.7   Projected
2029       52597.8   18830.0   Projected
2030       56069.3   20184.9   Projected
2031       59321.3   21355.7   Projected


In [6]:
import yfinance as yf


crm = yf.Ticker("CRM")
beta = crm.info.get('beta')
print(f"Salesforce Beta: {beta}")

Salesforce Beta: 1.152


In [7]:
# market data for WACC inputs
risk_free_rate = 0.0428  # 10-year US Treasury yield as of mid-2026
erp = 0.0453  # Damodaran ERP estimate (updated Jan 2026, stern.nyu.edu)

# cost of equity via CAPM
cost_of_equity = risk_free_rate + beta * erp
print(f"Risk Free Rate: {risk_free_rate:.2%}")
print(f"Equity Risk Premium: {erp:.2%}")
print(f"Beta: {beta}")
print(f"Cost of Equity: {cost_of_equity:.2%}")

Risk Free Rate: 4.28%
Equity Risk Premium: 4.53%
Beta: 1.152
Cost of Equity: 9.50%


In [8]:
# pull debt and interest expense from yfinance
balance_sheet = crm.balance_sheet
income_stmt = crm.financials

# most recent year's figures
total_debt = balance_sheet.loc['Total Debt'].iloc[0] / 1_000_000
interest_expense = abs(income_stmt.loc['Interest Expense'].iloc[0]) / 1_000_000

# effective cost of debt = interest paid / total debt outstanding
cost_of_debt_pretax = interest_expense / total_debt

# debt interest is tax deductible, so after-tax cost is lower
normalised_tax_rate = 0.183
cost_of_debt_aftertax = cost_of_debt_pretax * (1 - normalised_tax_rate)

print(f"Total Debt: ${total_debt:,.1f}M")
print(f"Interest Expense: ${interest_expense:,.1f}M")
print(f"Pre-tax Cost of Debt: {cost_of_debt_pretax:.2%}")
print(f"After-tax Cost of Debt: {cost_of_debt_aftertax:.2%}")

KeyError: 'Interest Expense'

In [9]:
# check what income statement items are available
print(income_stmt.index.tolist())


['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Write Off', 'Restructuring And Mergern Acquisition', 'Gain On Sale Of Security', 'Operating Income', 'Operating Expense', 'Depreciation Amortization Depletion Income Statement', 'Depreciation And Amortization In Inc

In [10]:
# interest expense from cash flow statement
cashflow = crm.cashflow
interest_expense = abs(cashflow.loc['Interest Paid Supplemental Data'].iloc[0]) / 1_000_000
total_debt = balance_sheet.loc['Total Debt'].iloc[0] / 1_000_000

# effective cost of debt
cost_of_debt_pretax = interest_expense / total_debt
cost_of_debt_aftertax = cost_of_debt_pretax * (1 - normalised_tax_rate)

print(f"Total Debt: ${total_debt:,.1f}M")
print(f"Interest Paid: ${interest_expense:,.1f}M")
print(f"Pre-tax Cost of Debt: {cost_of_debt_pretax:.2%}")
print(f"After-tax Cost of Debt: {cost_of_debt_aftertax:.2%}")

NameError: name 'normalised_tax_rate' is not defined

In [11]:
# interest expense from cash flow statement
cashflow = crm.cashflow
interest_expense = abs(cashflow.loc['Interest Paid Supplemental Data'].iloc[0]) / 1_000_000
total_debt = balance_sheet.loc['Total Debt'].iloc[0] / 1_000_000

# same normalised tax rate we calculated in data acquisition
normalised_tax_rate = 0.183

# effective cost of debt
cost_of_debt_pretax = interest_expense / total_debt
cost_of_debt_aftertax = cost_of_debt_pretax * (1 - normalised_tax_rate)

print(f"Total Debt: ${total_debt:,.1f}M")
print(f"Interest Paid: ${interest_expense:,.1f}M")
print(f"Pre-tax Cost of Debt: {cost_of_debt_pretax:.2%}")
print(f"After-tax Cost of Debt: {cost_of_debt_aftertax:.2%}")

Total Debt: $17,176.0M
Interest Paid: $276.0M
Pre-tax Cost of Debt: 1.61%
After-tax Cost of Debt: 1.31%


In [12]:
# Salesforce rated A2/A by Moody's/S&P
# investment grade spread of ~80bps over risk free rate is appropriate
# this better reflects current borrowing cost than historical interest/debt ratio

total_debt = balance_sheet.loc['Total Debt'].iloc[0] / 1_000_000
normalised_tax_rate = 0.183
risk_free_rate = 0.0428

cost_of_debt_pretax = risk_free_rate + 0.0080
cost_of_debt_aftertax = cost_of_debt_pretax * (1 - normalised_tax_rate)

print(f"Total Debt: ${total_debt:,.1f}M")
print(f"Credit Rating: A2/A (investment grade)")
print(f"Pre-tax Cost of Debt: {cost_of_debt_pretax:.2%}")
print(f"After-tax Cost of Debt: {cost_of_debt_aftertax:.2%}")

Total Debt: $17,176.0M
Credit Rating: A2/A (investment grade)
Pre-tax Cost of Debt: 5.08%
After-tax Cost of Debt: 4.15%


In [13]:
# capital structure weights
# equity weight = market cap / (market cap + total debt)
market_cap = crm.info.get('marketCap') / 1_000_000

total_capital = market_cap + total_debt
weight_equity = market_cap / total_capital
weight_debt = total_debt / total_capital

# WACC = (E/V x cost of equity) + (D/V x after-tax cost of debt)
wacc = (weight_equity * cost_of_equity) + (weight_debt * cost_of_debt_aftertax)

print(f"Market Cap: ${market_cap:,.1f}M")
print(f"Total Debt: ${total_debt:,.1f}M")
print(f"Total Capital: ${total_capital:,.1f}M")
print(f"\nEquity Weight: {weight_equity:.1%}")
print(f"Debt Weight: {weight_debt:.1%}")
print(f"\nCost of Equity: {cost_of_equity:.2%}")
print(f"After-tax Cost of Debt: {cost_of_debt_aftertax:.2%}")
print(f"\nWACC: {wacc:.2%}")

Market Cap: $156,404.4M
Total Debt: $17,176.0M
Total Capital: $173,580.4M

Equity Weight: 90.1%
Debt Weight: 9.9%

Cost of Equity: 9.50%
After-tax Cost of Debt: 4.15%

WACC: 8.97%
